# Session 1 - Messages API, Streaming and SDK Setup

Read the explanation, run the cell, look at the output. Each section maps to a block of the session:

| # | Section | What you will see |
|---|---------|-------------------|
| 0 | Setup | Load the API key, create the client |
| 1 | Messages API | Required fields, response object, content blocks, system prompt, `max_tokens`, `stop_reason`, tools, thinking |
| 2 | SDK setup and multi-turn state | Auth, helper functions, history, token growth, auth errors |
| 3 | Streaming | Non-streaming vs streaming, raw events, `text_stream`, partial tool JSON |
| 4 | Lab | One streaming, multi-turn chat client that pulls it all together |

<br>

> Outputs differ on every run (the model is not deterministic). The **structure** is what matters.

## 0. Setup

1. `pip install anthropic python-dotenv`

The SDK reads `ANTHROPIC_API_KEY` from the environment, so you never write the key in code.

In [ ]:
import os

# os.environ["ANTHROPIC_API_KEY"] = "sk-ant...."  # replace with your key.

In [ ]:
%pip install anthropic python-dotenv requests  
from dotenv import load_dotenv
load_dotenv()

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
import json, time
# %pip install anthropic python-dotenv
from anthropic import Anthropic

client = Anthropic()               # picks up ANTHROPIC_API_KEY automatically
MODEL = "claude-haiku-5-5"         # a current model ID; small and cheap for a classroom

print("API key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))   # prints True/False, never the key


def text_of(message):
    """Join the text blocks. content[0] is not always text: it can be a thinking or tool_use block."""
    return "".join(b.text for b in message.content if b.type == "text")

API key loaded: True


### Warm-up: does Claude remember earlier requests?

Two **separate** requests. If the API kept a session, the second would know the name.

In [35]:
r1 = client.messages.create(
    model=MODEL, max_tokens=2000,
    messages=[{"role": "user", "content": "My name is Vijay. Remember it."}],
)
print("Request 1 ->", text_of(r1))

r2 = client.messages.create(                       # a brand-new request, no history
    model=MODEL, max_tokens=2000,
    messages=[{"role": "user", "content": "What is my name?"}],
)
print("\nRequest 2 ->", text_of(r2))

Request 1 -> I'll remember that your name is Vijay. Keep in mind that I don't retain memory between separate conversations, so if you come back later, you may need to tell me again. Within this conversation, I'll keep it in mind.

Request 2 -> I don't know your name. This is the start of our conversation, and I don't have access to any personal information about you unless you share it with me.

If you tell me your name, I'm happy to use it in our conversation.


**Takeaway:** every request is independent. The API is **stateless**; you keep the message list and resend it (section 3).

## 1. Messages API

### 1.1 The request lifecycle

```
Client (browser/app) --1--> Your server --2--> Anthropic API --3--> model
Client (browser/app) <--5-- Your server <--4-- Anthropic API <------+
```

**Why a server?** The API key is secret. Code that runs on the client can be inspected, so a key placed there can be extracted and used on your account. The key stays on your server.

### 1.2 Required fields

| Field | Purpose |
|-------|---------|
| API key | Authenticates the request (SDK handles the `x-api-key` header) |
| `model` | Which Claude model |
| `messages` | The conversation: alternating `user` / `assistant` messages |
| `max_tokens` | A **safety ceiling** on generated tokens, *not* a target |

Your first call:

In [36]:
message = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=[{"role": "user", "content": "What is quantum computing? Answer in one sentence."}],
)
print(text_of(message))

Quantum computing is a type of computation that uses quantum-mechanical phenomena such as superposition and entanglement to process information in ways that can solve certain problems much faster than classical computers.


## 1.3 The response object

`message` is a `Message` object. Inspect the fields:

In [37]:
print("id           :", message.id)             # starts with msg_
print("type         :", message.type)           # always "message"
print("role         :", message.role)           # always "assistant"
print("model        :", message.model)
print("stop_reason  :", message.stop_reason)
print("stop_sequence:", message.stop_sequence)  # None unless a custom stop string matched
print("usage        :", message.usage.input_tokens, "in /", message.usage.output_tokens, "out")

id           : msg_011CgjZqJiRh8ttVDEeVDLyy
type         : message
role         : assistant
model        : claude-haiku-5-5
stop_reason  : end_turn
stop_sequence: None
usage        : 25 in / 62 out


In [38]:
# The full raw structure, as the API returns it
print(message.model_dump_json(indent=2))

{
  "id": "msg_011CgjZqJiRh8ttVDEeVDLyy",
  "container": null,
  "content": [
    {
      "citations": null,
      "text": "Quantum computing is a type of computation that uses quantum-mechanical phenomena such as superposition and entanglement to process information in ways that can solve certain problems much faster than classical computers.",
      "type": "text"
    }
  ],
  "model": "claude-haiku-5-5",
  "role": "assistant",
  "stop_details": null,
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "global",
    "input_tokens": 25,
    "output_tokens": 62,
    "output_tokens_details": {
      "thinking_tokens": 0
    },
    "server_tool_use": null,
    "service_tier": "standard",
    "speed": "standard"
  },
  "diagnostics": null
}


Note `content` is a **list of blocks**, not a string. `usage` is what you are billed on (input + output tokens).

### 1.4 Content blocks

Never assume `content[0]` is text. Loop over the blocks and check `block.type`.

| Block `type` in a response | Meaning |
|----------------------------|---------|
| `text` | Generated text |
| `tool_use` | Claude wants to call **your** tool (`id`, `name`, `input`) |
| `thinking` | Claude's reasoning (extended thinking enabled) |
| `server_tool_use` + `*_tool_result` | A tool **Anthropic** ran (web search, code execution, ...) |

In a request, a plain string is shorthand for one `text` block. You can also send `image`, `document` and `tool_result` blocks.

> **Why not just `content[0].text`?** Current models can return a `thinking` block **before** the `text` block. Then `content[0]` is a `ThinkingBlock`, and `.text` fails with `'ThinkingBlock' object has no attribute 'text'`. The `text_of(message)` helper from the setup cell joins only the `text` blocks, so it works either way.

In [39]:
def render(message):
    """Return the visible text, handling every block type safely."""
    parts = []
    for block in message.content:
        if block.type == "text":
            parts.append(block.text)
        elif block.type == "tool_use":
            parts.append(f"[tool_use: {block.name}({block.input})]")
        elif block.type == "thinking":
            parts.append(f"[thinking: {block.thinking[:60]}...]")
        else:
            parts.append(f"[{block.type}]")
    return "\n".join(parts)

print(render(message))

Quantum computing is a type of computation that uses quantum-mechanical phenomena such as superposition and entanglement to process information in ways that can solve certain problems much faster than classical computers.


### 1.5 Roles and the system prompt

* Only two roles appear in `messages`: **`user`** and **`assistant`**, and they **alternate**.
* The system prompt is **not a role**. It is the top-level `system` parameter.

One question, asked with and without a system prompt:

In [40]:
question = [{"role": "user", "content": "How do I solve 5x + 2 = 3 for x?"}]

a = client.messages.create(model=MODEL, max_tokens=2000, messages=question)
b = client.messages.create(
    model=MODEL, max_tokens=2000, messages=question,
    system="You are a patient math tutor. Do not directly answer. Guide the student step by step.",
)

print("A. No system prompt:\n", text_of(a))
print("\nB. With tutor system prompt:\n", text_of(b))

A. No system prompt:
 Subtract 2 from both sides, then divide both sides by 5:

1. Subtract 2 from both sides:
 5x + 2 − 2 = 3 − 2
 5x = 1

2. Divide both sides by 5:
 5x ÷ 5 = 1 ÷ 5
 x = 1/5 (or 0.2)

**Check:** 5(1/5) + 2 = 1 + 2 = 3 ✓

B. With tutor system prompt:
 Let's work through it together. The goal is to get x alone on one side of the equation.

**Step 1:** Look at the "+ 2" on the left side. What operation would undo adding 2? Try subtracting 2 from both sides, and tell me what the equation becomes.


The system prompt changes **behaviour**, not the question.

> **Gotcha:** the API rejects `system=None`. Only include the parameter when you have a value:

In [41]:
def chat(messages, system=None, max_tokens=1000):
    params = {"model": MODEL, "max_tokens": max_tokens, "messages": messages}
    if system:                         # add only when set - never pass system=None
        params["system"] = system
    return text_of(client.messages.create(**params))

print(chat([{"role": "user", "content": "Name one planet."}]))

Mars.


### 1.6 `max_tokens` and `stop_reason`

`max_tokens` is a **ceiling**. If it is hit, the reply is **cut off silently** - there is no error, so you must check `stop_reason` yourself.

In [42]:
def show(message):
    print(text_of(message))
    print(f"-> stop_reason: {message.stop_reason} | output_tokens: {message.usage.output_tokens}")

print("Normal (max_tokens=1000):")
show(client.messages.create(model=MODEL, max_tokens=1000, messages=question))

print("\nTruncated (max_tokens=15):")
show(client.messages.create(model=MODEL, max_tokens=15, messages=question))

Normal (max_tokens=1000):
Subtract 2 from both sides, then divide both sides by 5:

1. Subtract 2 from both sides:
   5x + 2 − 2 = 3 − 2
   5x = 1

2. Divide both sides by 5:
   5x ÷ 5 = 1 ÷ 5
   x = 1/5 (or 0.2)

**Check:** 5(1/5) + 2 = 1 + 2 = 3 ✓
-> stop_reason: end_turn | output_tokens: 147

Truncated (max_tokens=15):
Subtract 2 from both sides, then divide
-> stop_reason: max_tokens | output_tokens: 15


Spot the difference: the second reply stops mid-sentence, `stop_reason` is `max_tokens`, and `output_tokens` equals the limit.

> On a model that thinks first, the reply can even be **empty**: thinking tokens count toward `max_tokens`, so all 15 were spent before any text. `stop_reason` is still `max_tokens`, which is exactly the signal to check.

**Custom stop sequences** end generation at a string you choose. The stop string itself is *not* included in the text.

In [43]:
m = client.messages.create(
    model=MODEL, max_tokens=2000,
    messages=[{"role": "user", "content": "List three required request fields, then write ###."}],
    stop_sequences=["###"],
)
print(repr(text_of(m)))
print("stop_reason  :", m.stop_reason)       # "stop_sequence"
print("stop_sequence:", m.stop_sequence)     # "###"

'1. Method (e.g., GET, POST)\n2. URL (or request path)\n3. Host header (required in HTTP/1.1)\n\n'
stop_reason  : stop_sequence
stop_sequence: ###


| `stop_reason` | Meaning | What your code should do |
|---------------|---------|--------------------------|
| `end_turn` | Finished naturally | Use the response |
| `max_tokens` | Hit your limit - **truncated** | Raise the limit or continue |
| `stop_sequence` | Hit your custom stop string | Use the text up to the stop point |
| `tool_use` | Claude wants to call a tool | Run it, send back a `tool_result` |
| `pause_turn` | A server-tool turn paused | Send the response back to let it continue |
| `refusal` | Declined for safety reasons | Show a fallback; do not retry unchanged |

In [44]:
NEXT_STEP = {
    "end_turn":      "Use the response.",
    "max_tokens":    "Truncated! Raise max_tokens or continue the generation.",
    "stop_sequence": "Use the text up to the stop point.",
    "tool_use":      "Run the tool and reply with a tool_result.",
    "pause_turn":    "Send the response back as-is so Claude can continue.",
    "refusal":       "Show a fallback message; do not retry unchanged.",
}

def next_step(message):
    return NEXT_STEP.get(message.stop_reason, f"Unknown stop_reason: {message.stop_reason}")

print(next_step(m))

Use the text up to the stop point.


### 1.7 Other request parameters

| Parameter | What it does |
|-----------|--------------|
| `temperature` | Randomness, `0.0`-`1.0` (default `1.0`). Session 2 covers this in depth |
| `top_p` / `top_k` | Alternative sampling controls. Change `temperature` **or** `top_p`, not both |
| `stop_sequences` | Custom strings that stop generation |
| `stream` | `True` returns server-sent events (section 4) |
| `metadata` | `{"user_id": "..."}` opaque ID for abuse tracking. **No personal data** |
| `tools` / `tool_choice` | Tool definitions and how Claude may pick them |
| `thinking` | Extended thinking: `{"type": "enabled", "budget_tokens": N}` |

Some newer models restrict sampling parameters, so this cell is wrapped to fail gracefully.

In [45]:
try:
    m = client.messages.create(
        model=MODEL, max_tokens=2000,
        system="You are a concise API instructor.",
        messages=[{"role": "user", "content": "In one line, what does temperature control?"}],
        temperature=0.3,
        metadata={"user_id": "learner-042"},
    )
    print(text_of(m))
except Exception as e:      # e.g. a model that rejects this parameter
    print(type(e).__name__, "->", e)

BadRequestError -> Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': '`temperature` is deprecated for this model.'}, 'request_id': 'req_011CgjZrEXgzJBR8BYQAtaRc'}


### 1.8 Tools: the `tool_use` block

When Claude wants to call a function, the response contains **two blocks** (a `text` block and a `tool_use` block) and `stop_reason == "tool_use"`. **Claude never runs your code.** You run the tool and reply with a `tool_result` that references the same `id`.

Here we only look at the response.

In [46]:
tools = [{
    "name": "get_weather",
    "description": "Get the current weather for a city.",
    "input_schema": {
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"],
    },
}]

m = client.messages.create(
    model=MODEL, max_tokens=2000, tools=tools, tool_choice={"type": "auto"},
    messages=[{"role": "user", "content": "What's the weather in Bengaluru?"}],
)

print("stop_reason:", m.stop_reason)
for block in m.content:
    print(" -", block.type, "|", block.model_dump(exclude_none=True))

stop_reason: tool_use
 - tool_use | {'id': 'toolu_0168F29155Z4HE37siz68UzS', 'caller': {'type': 'direct'}, 'input': {'city': 'Bengaluru'}, 'name': 'get_weather', 'type': 'tool_use'}


> **Rule of thumb:** `tool_use` = *you* run the tool. `server_tool_use` = *Anthropic* already ran it and the result is in the same response.

### 1.9 Extended thinking

With thinking on, the response gains a `thinking` block that comes **before** the `text` block. Current models use **adaptive** thinking (`{"type": "adaptive"}`): the model decides how much to think. The older `budget_tokens` setting is rejected with a 400 on these models.

* Current models think **by default**, which is why `content[0]` is not always text.
* Thinking tokens count toward `output_tokens` **and** toward `max_tokens`, so leave room for both.
* `display: "summarized"` makes the reasoning readable. The default (`"omitted"`) returns an empty `thinking` field. It is billed either way.
* If you continue the conversation, pass thinking blocks back **unmodified**.

In [47]:
try:
    m = client.messages.create(
        model=MODEL, max_tokens=4000,
        thinking={"type": "adaptive", "display": "summarized"},
        messages=[{"role": "user", "content": "Is 391 a prime number?"}],
    )
    for block in m.content:
        print(f"[{block.type}]", getattr(block, "thinking", None) or getattr(block, "text", ""))
    print("output_tokens (includes thinking):", m.usage.output_tokens)
except Exception as e:
    print(type(e).__name__, "->", e)

[thinking] I'm checking whether 391 is prime, and it factors as 17 × 23, so it isn't.


[text] No, 391 is not prime. It equals 17 × 23.
output_tokens (includes thinking): 56


## 2. SDK setup and multi-turn state

### 2.1 Authentication

* The SDK reads **`ANTHROPIC_API_KEY`** from the environment. Prefer this over `Anthropic(api_key=...)`.
* Keep the key in `.env` (loaded with `python-dotenv`) and **never commit it**.
* Official SDKs exist for Python, TypeScript, JavaScript, Go and Ruby. Plain HTTP works too.
* A leaked key must be **revoked** in the console and replaced.
* A key is shown only once when created, so a lost key must be replaced too.

Two common errors you can reproduce safely (they use a fake key, so your real one is never touched):

In [48]:
from anthropic import AuthenticationError

# Error 1: a request with a WRONG key -> HTTP 401
bad_client = Anthropic(api_key="sk-ant-this-key-is-fake")
try:
    bad_client.messages.create(model=MODEL, max_tokens=10,
                               messages=[{"role": "user", "content": "hi"}])
except AuthenticationError as e:
    print("401 AuthenticationError:", e.message)

401 AuthenticationError: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CgjZrSciGSxe361pqq4Nf'}


The other classic: **no key at all** raises `TypeError: Could not resolve authentication method...` when the client is created or used. The fix is almost always to call `load_dotenv()` **before** `Anthropic()` and to check the `.env` location.

### 2.2 The API is stateless

Claude **does not store your conversation**. To hold a multi-turn chat you must:

1. Keep a list of all messages in **your** code.
2. Send the **complete history** with every request.

### 2.3 Helper functions

(The tiny `chat` helper from section 2 returns only the text. Here we also manage history.)

In [49]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

messages = []

add_user_message(messages, "Define quantum computing in one sentence.")
answer = chat(messages)
add_assistant_message(messages, answer)          # do not forget this step!

add_user_message(messages, "Write another sentence.")
final_answer = chat(messages)                    # Claude now has the context

print("Turn 1:", answer)
print("Turn 2:", final_answer)

Turn 1: Quantum computing is a computing paradigm that uses quantum-mechanical phenomena such as superposition, entanglement, and interference to process information in ways that can solve certain problems more efficiently than classical computers.
Turn 2: Unlike classical bits, which are always either 0 or 1, quantum bits (qubits) can exist in a combination of both states at once, allowing quantum computers to explore many computational paths simultaneously.


What the history looks like on the wire:

In [50]:
print(json.dumps(messages, indent=2))

[
  {
    "role": "user",
    "content": "Define quantum computing in one sentence."
  },
  {
    "role": "assistant",
    "content": "Quantum computing is a computing paradigm that uses quantum-mechanical phenomena such as superposition, entanglement, and interference to process information in ways that can solve certain problems more efficiently than classical computers."
  },
  {
    "role": "user",
    "content": "Write another sentence."
  }
]


### 2.4 The same follow-up WITHOUT history

This is the symptom behind exam question Q3.

In [51]:
print(chat([{"role": "user", "content": "Write another sentence."}]))

The lighthouse keeper watched the storm roll across the bay, counting the seconds between each flash of lightning.


Claude has no idea what "another" refers to, so it writes about something unrelated.

### 2.5 Input tokens grow every turn

Each request resends everything, so cost and latency grow with the length of the chat.

In [52]:
history = []
for turn in ["What is a qubit?", "And superposition?", "Give me an analogy."]:
    add_user_message(history, turn)
    m = client.messages.create(model=MODEL, max_tokens=2000, messages=history)
    add_assistant_message(history, text_of(m))
    print(f"{len(history)} messages -> {m.usage.input_tokens} input tokens")

2 messages -> 15 input tokens
4 messages -> 578 input tokens
6 messages -> 1272 input tokens


| Consequence of statelessness | Why it matters |
|------------------------------|----------------|
| History grows each turn | Input tokens, cost and latency grow with it |
| Your app owns the state | Drop a message and Claude "forgets" it |
| Long chats can hit the context limit | Plan to trim or summarise old turns |
| You can edit history | Handy for testing; a mistake here changes behaviour |


## 3. Streaming

A full reply can take 10-30 seconds. Without streaming the user stares at a spinner. With streaming, text appears as it is generated. It is still **one request**; the server forwards chunks as they arrive.

| | Non-streaming | Streaming |
|---|---------------|-----------|
| Call | `messages.create(...)` | `messages.stream(...)` or `stream=True` |
| Response | One JSON `Message` | A series of events |
| User sees text | After the **whole** reply is done | As each chunk arrives |
| `stop_reason` | In the response body | In the `message_delta` event |
| Best for | Short replies, batch jobs | Chat UIs, long replies |

### 3.1 Feel the difference

Same prompt, both ways. We measure **time to first text** and **total time**.

In [53]:
prompt = [{"role": "user", "content": "Write a 120-word description of a fictional database."}]

# Non-streaming: nothing to show until the whole reply is done
t0 = time.time()
full = client.messages.create(model=MODEL, max_tokens=2000, messages=prompt)
t_total = time.time() - t0
print(f"Non-streaming: first text visible after {t_total:.1f}s (= total) | stop_reason={full.stop_reason}")

# Streaming: the first text chunk arrives as soon as the model starts writing
t0, first = time.time(), None
with client.messages.stream(model=MODEL, max_tokens=2000, messages=prompt) as stream:
    for text in stream.text_stream:
        if first is None:
            first = time.time() - t0
    final = stream.get_final_message()
total = time.time() - t0

if first is None:
    print(f"Streaming    : no text arrived (stop_reason={final.stop_reason}). "
          "The token budget was probably used up by thinking: raise max_tokens.")
else:
    print(f"Streaming    : first text visible after {first:.1f}s, total {total:.1f}s")

Non-streaming: first text visible after 6.1s (= total) | stop_reason=end_turn
Streaming    : first text visible after 6.3s, total 6.9s


The total time is about the same, but with streaming the user starts reading almost immediately.

> On a model that **thinks first**, "first text" also includes the thinking time, and the limit has to be large enough for the thinking **and** the answer. That is why this cell uses `max_tokens=2000`.

### 3.2 The raw event stream

With `stream=True` you receive events and parse them yourself.

| Event | Meaning |
|-------|---------|
| `message_start` | A new message begins (empty `content`, `stop_reason` is `None`) |
| `content_block_start` | A new block begins (text, tool use, ...) |
| `content_block_delta` | A chunk of the block. **This carries the text you display** |
| `content_block_stop` | The block is complete |
| `message_delta` | Top-level changes - the final **`stop_reason`** and usage |
| `message_stop` | The message is complete |

The API may also send `ping` and `error` events. Handle them without crashing.

In [54]:
stream = client.messages.create(
    model=MODEL, max_tokens=2000, stream=True,
    messages=[{"role": "user", "content": "Say hello in two words."}],
)

for event in stream:
    if event.type == "content_block_delta":
        d = event.delta
        field = {"text_delta": "text", "thinking_delta": "thinking", "input_json_delta": "partial_json"}.get(d.type)
        print(f"{event.type:22} {d.type}: {getattr(d, field) if field else '...'!r}")
    elif event.type == "message_delta":
        print(f"{event.type:22} stop_reason={event.delta.stop_reason}, output_tokens={event.usage.output_tokens}")
    else:
        print(event.type)

message_start
content_block_start
content_block_delta    text_delta: 'Hello there'
content_block_delta    text_delta: '!'
content_block_stop
message_delta          stop_reason=end_turn, output_tokens=7
message_stop


Order for a simple text reply:

```
message_start
  content_block_start
    content_block_delta   (x many)
  content_block_stop
message_delta             <- stop_reason arrives here
message_stop
```

Delta types: `text_delta` (text), `input_json_delta` (partial tool-call JSON), `thinking_delta` (extended thinking). On a model that thinks first, you will see `thinking_delta` (and a `signature_delta`) events **before** the first `text_delta`, so read the delta's type before reading its text.

### 3.3 The helper: `messages.stream`

Use the helper unless you need individual events. `text_stream` yields just the text, and `get_final_message()` returns the **assembled** message (for saving to a DB or appending to the history).

In [55]:
with client.messages.stream(
    model=MODEL, max_tokens=2000,
    messages=[{"role": "user", "content": "Explain streaming in two sentences."}],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)           # flush=True or the terminal buffers it

    final = stream.get_final_message()             # inside the with-block

print()
print("stop_reason:", final.stop_reason, "| output_tokens:", final.usage.output_tokens)
print("assembled  :", text_of(final)[:60] + "...")

Streaming is the continuous delivery of data, such as audio, video, or text, in a steady flow so it can be played or processed as it arrives, rather than waiting for the entire file to download first. This approach allows users to begin consuming content almost immediately while the rest is still being transmitted in the background.
stop_reason: end_turn | output_tokens: 95
assembled  : Streaming is the continuous delivery of data, such as audio,...


### 3.4 Partial tool-call JSON

When Claude calls a tool during a stream, the arguments arrive as **fragments** (`input_json_delta`). Each fragment on its own is **not valid JSON**. Accumulate, then parse after `content_block_stop`.

In [56]:
fragments, tool_name = [], None
stream = client.messages.create(
    model=MODEL, max_tokens=2000, stream=True, tools=tools,
    tool_choice={"type": "tool", "name": "get_weather"},     # force the tool call for the demo
    messages=[{"role": "user", "content": "Weather in Chennai?"}],
)

for event in stream:
    if event.type == "content_block_start" and event.content_block.type == "tool_use":
        tool_name = event.content_block.name
    elif event.type == "content_block_delta" and event.delta.type == "input_json_delta":
        fragments.append(event.delta.partial_json)
        print("fragment:", repr(event.delta.partial_json))
    elif event.type == "content_block_stop" and fragments:
        break

print("\nTool:", tool_name)
print("Joined:", "".join(fragments))
print("Parsed:", json.loads("".join(fragments)))             # safe only now

fragment: ''
fragment: '{"city"'
fragment: ': "Chen'
fragment: 'nai"}'

Tool: get_weather
Joined: {"city": "Chennai"}
Parsed: {'city': 'Chennai'}


### 3.5 Partial output pitfalls

* **Do not treat partial text as final.** A stream can end early (error, or a `max_tokens` stop). Read `stop_reason` *after* the stream.
* **Tool-call JSON arrives in fragments.** Parse only after the block stops.
* **Flush output** (`flush=True`) or streaming looks broken in a terminal.

## 4. Lab - a streaming, multi-turn chat client

Everything from sections 1-3 in one function: **system prompt + history + streaming + `stop_reason`**.

| Capability | Where it came from |
|------------|--------------------|
| `messages.stream` + `text_stream` | Section 3 |
| `system` only when set | Section 1.5 |
| `max_tokens` handling | Section 1.6 |
| History kept in sync | Section 2 |

In [57]:
SYSTEM = "You are a concise technical assistant."     # experiment with this

def stream_reply(messages, system=SYSTEM, max_tokens=1000):
    params = {"model": MODEL, "max_tokens": max_tokens, "messages": messages}
    if system:
        params["system"] = system
    with client.messages.stream(**params) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
        final = stream.get_final_message()

    print(f"\n[stop_reason: {final.stop_reason}]")
    if final.stop_reason == "max_tokens":
        print("(reply was cut off - raise max_tokens)")
    return text_of(final)


def run_chat(turns, system=SYSTEM, max_tokens=1000, keep_history=True):
    """Scripted chat so it runs in a notebook. keep_history=False shows what breaks."""
    messages = []
    for user_text in turns:
        print(f"\nYou: {user_text}\nClaude: ", end="")
        add_user_message(messages, user_text)
        reply = stream_reply(messages, system, max_tokens)
        if keep_history:
            add_assistant_message(messages, reply)      # keep the history in sync
        else:
            messages.pop()                              # also drop the user turn -> every turn is fresh
    return messages

history = run_chat(["What is a token?", "Give me an example."])


You: What is a token?
Claude: A token is a basic unit of text that a language model reads and generates. Instead of processing raw characters or whole words, models split text into tokens, which can be whole words, parts of words, punctuation marks, or spaces. For example, "unbelievable" might be split into "un", "believ", and "able," while a common word like "the" is usually a single token.

Tokens matter because models have limits measured in tokens (such as context window size), and pricing for many AI APIs is based on the number of tokens processed.
[stop_reason: end_turn]

You: Give me an example.
Claude: Here's a concrete example using the sentence:

**"I can't believe it's already 2024!"**

A tokenizer might split it like this:

| Token | Text |
|---|---|
| 1 | "I" |
| 2 | " can" |
| 3 | "'t" |
| 4 | " believe" |
| 5 | " it" |
| 6 | "'s" |
| 7 | " already" |
| 8 | " 202" |
| 9 | "4" |
| 10 | "!" |

That's 10 tokens for a 10-word-ish sentence. Notice that:

- Spaces are usually 

**Check:** the second answer builds on the first (the history works) and `[stop_reason: end_turn]` printed after each reply.

### Experiments

Run each one and compare with the baseline above.

In [58]:
# 1. Different system prompt
_ = run_chat(["What is a token?"], system="You are a pirate. Answer in two short sentences.")


You: What is a token?
Claude: Arr, a token be a small chunk o' text, like a word, part of a word, or a punctuation mark, that language models read and process. Ye might think o' it as a piece o' treasure that the AI counts and trades with.
[stop_reason: end_turn]


In [59]:
# 2. Tiny max_tokens -> truncated, stop_reason == "max_tokens"
_ = run_chat(["Explain how the internet works in detail."], max_tokens=30)


You: Explain how the internet works in detail.
Claude: 
[stop_reason: max_tokens]
(reply was cut off - raise max_tokens)


In [60]:
# 3. History NOT kept -> the follow-up loses context
_ = run_chat(["What is a token?", "Give me an example."], keep_history=False)


You: What is a token?
Claude: A token is a basic unit of text that a language model reads and generates. Text is split into tokens by a tokenizer, and a token can be a whole word, part of a word, a punctuation mark, or a space or byte sequence. For example, "unbelievable" might be split into "un", "believ", and "able," while a common word like "the" is usually a single token.

On average, one token in English is roughly 4 characters or about three-quarters of a word. Tokens matter because models process input and output in tokens, and API usage and context-window limits are measured in tokens.
[stop_reason: end_turn]

You: Give me an example.
Claude: I'm not sure what you'd like an example of, since this is the start of our conversation. Could you tell me the topic or paste the text or code you have in mind?

In the meantime, here's an example of a common request, a Python function with a docstring:

```python
def celsius_to_fahrenheit(c: float) -> float:
    """Convert a temperature 

In [61]:
# 4. Token usage grows with history (stretch goal)
msgs = []
for q in ["What is a token?", "Give me an example.", "Why does it matter for cost?"]:
    add_user_message(msgs, q)
    with client.messages.stream(model=MODEL, max_tokens=2000, system=SYSTEM, messages=msgs) as s:
        for _ in s.text_stream:
            pass
        fm = s.get_final_message()
    add_assistant_message(msgs, text_of(fm))
    print(f"turn {len(msgs)//2}: in={fm.usage.input_tokens} out={fm.usage.output_tokens}")

turn 1: in=25 out=133
turn 2: in=166 out=146
turn 3: in=324 out=269


## 5. Wrap-up

### Cheat sheet

| Topic | Remember |
|-------|----------|
| Request | `model` + `messages` + `max_tokens` required. `system` optional, top-level, never `None` |
| Roles | `user` and `assistant`, alternating. System is a parameter, not a role |
| Response | `content` is a **list of blocks**. Also `stop_reason` and `usage` |
| `stop_reason` | `end_turn` / `max_tokens` (truncated) / `stop_sequence` / `tool_use` / `pause_turn` / `refusal` |
| Streaming events | `message_start` > `content_block_start` > `content_block_delta` x N > `content_block_stop` > `message_delta` (stop_reason) > `message_stop` |
| Streaming code | `client.messages.stream()` + `.text_stream` + `.get_final_message()` |
| State | Stateless. Append every user **and** assistant message and resend the whole history |
| Security | Key in `ANTHROPIC_API_KEY` / `.env`, never in client code or git. Requests go through your server |